# Contexto e limpeza de recursos

Leia um arquivo pequeno e preveja se continuará aberto após sair do bloco. Retome [with e arquivos](https://devrgs.github.io/Aprendendo-linguagens/#/python/arquivos/caminhos-e-texto), [exceções](https://devrgs.github.io/Aprendendo-linguagens/#/python/arquivos/excecoes) e [yield](https://devrgs.github.io/Aprendendo-linguagens/#/python/protocolos/geradores).

Um recurso tem entrada, uso e liberação. `with` aciona o gerente que prepara e finaliza esse uso. O gerenciador pode lidar com a saída por erro. `try/finally` executa a limpeza ao sair do bloco, inclusive se uma exceção se propagar. Com `contextlib.contextmanager`, código antes de `yield` prepara, o valor emitido pode ser usado por `as`, e `finally` executa a liberação.

Os arquivos de E1 e E2 estão nas pastas do roteiro. E2 e E3 levantam erros intencionais e os capturam após a saída do contexto. Qual saída mostraria que a limpeza aconteceu antes da captura? Depois, veja [testes unitários](https://devrgs.github.io/Aprendendo-linguagens/#/python/qualidade/testes-unitarios).

Execute as células na ordem com **Shift+Enter**. A preparação cria uma pasta nova na área de arquivos do caderno. Baixe o caderno editado e os resultados que quiser guardar.


In [ ]:
from pathlib import Path as _Path
from tempfile import mkdtemp as _mkdtemp
import os as _os
import sys as _sys
from importlib import invalidate_caches as _refresh_imports

# Auxiliares editáveis devem ser relidos sem aproveitar bytecode antigo.
_sys.dont_write_bytecode = True

_raiz = _Path.cwd()
_trabalho = _Path(_mkdtemp(prefix='python-gerentes-de-contexto-', dir=_raiz))


## E1 · Preveja e execute

Leia `amostra.txt` e imprima `arquivo.closed` dentro e fora de with. Preveja as duas respostas.


In [ ]:
_pasta = _trabalho / 'e1'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_arquivo = _pasta / 'amostra.txt'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('sessão\n', encoding='utf-8')
_refresh_imports()


In [ ]:
from pathlib import Path

with Path("amostra.txt").open(encoding="utf-8") as arquivo:
    print(arquivo.read().strip())
    print(arquivo.closed)
print(arquivo.closed)


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
sessão
False
True
```

O arquivo está aberto para leitura dentro do bloco e fechado após a saída. O nome permanece disponível para consultar seu estado.


## E2 · Altere ou depure

Levante ValueError intencional dentro do bloco após ler `amostra.txt`. Capture fora e verifique se o arquivo já está fechado.


In [ ]:
_pasta = _trabalho / 'e2'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_arquivo = _pasta / 'amostra.txt'
_arquivo.parent.mkdir(parents=True, exist_ok=True)
if not _arquivo.exists():
    _arquivo.write_text('sessão\n', encoding='utf-8')
_refresh_imports()


In [ ]:
from pathlib import Path

try:
    with Path("amostra.txt").open(encoding="utf-8") as arquivo:
        print(arquivo.read().strip())
        raise ValueError("dado inválido")
except ValueError as erro:
    print(erro)
    print("fechado:", arquivo.closed)


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Saída esperada

```text
sessão
dado inválido
fechado: True
```

O gerente fecha o arquivo durante a saída por exceção e a captura externa recebe o ValueError original.


## E3 · Resolva

Implemente um contexto que registre início e fim. Faça uma execução normal e outra que levante ValueError no bloco. Registre o fim em ambos os casos e confirme que a exceção observada fora é a original.

**Pista.** Coloque yield dentro de try e registre o fim em finally.

Saída esperada

```text
início / trabalho / fim
início / fim
mesma exceção: True
```


In [ ]:
_pasta = _trabalho / 'e3'
_pasta.mkdir(exist_ok=True)
_os.chdir(_pasta)
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]
if str(_pasta) not in _sys.path:
    _sys.path.insert(0, str(_pasta))
_refresh_imports()


In [ ]:
from contextlib import contextmanager

@contextmanager
def registrar(eventos):
    eventos.append("início")
    # Suspenda durante o bloco e registre o fim na saída.
    raise NotImplementedError

eventos = []
# Teste saídas normal e por ValueError.


In [ ]:
# Voltar à pasta do caderno para a etapa seguinte.
_os.chdir(_raiz)
if str(_pasta) in _sys.path:
    _sys.path.remove(str(_pasta))
# Liberar módulos locais para a próxima etapa importar sua própria versão.
for _nome, _modulo in list(_sys.modules.items()):
    _origem = getattr(_modulo, '__file__', None)
    if _origem and _Path(_origem).is_relative_to(_pasta):
        del _sys.modules[_nome]


Compare com a solução comentada na aula e faça o caso de borda antes de concluir.
